# 02 — Decision Tree (CART) From Scratch vs Sklearn
Implementasi CART from scratch, visualisasi tree, dan perbandingan dengan sklearn.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report

from src.utils import set_seed, save_model
from src.data import load_train, load_test
from src.cleaning import DataCleaner
from src.preprocessing import Preprocessor
from src.algorithms.decision_tree import DecisionTreeCART
from src.evaluation import cross_validate, macro_f1_score, classification_report_manual
from src.sklearn_baselines import get_sklearn_dtl
from src.visualization import plot_tree_structure
from src import config

set_seed(42)

In [ ]:
# Load & preprocess
train_df = load_train()
cleaner = DataCleaner()
train_clean = cleaner.fit_transform(train_df)
preprocessor = Preprocessor()
X_train, y_train = preprocessor.fit_transform(train_clean)
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')

In [ ]:
# Cross-validate CART from scratch
def dtl_factory():
    return DecisionTreeCART(
        max_depth=10, min_samples_split=10, min_samples_leaf=5,
        class_weight='balanced'
    )

dtl_cv = cross_validate(dtl_factory, X_train, y_train, n_folds=5)
print(f'CART From-Scratch:')
print(f'  Fold scores: {[f"{s:.4f}" for s in dtl_cv["fold_scores"]]}')
print(f'  Mean Macro F1: {dtl_cv["mean_f1"]:.4f} +/- {dtl_cv["std_f1"]:.4f}')

In [ ]:
# Cross-validate sklearn baseline
def dtl_sk_factory():
    return get_sklearn_dtl(max_depth=10)

dtl_sk_cv = cross_validate(dtl_sk_factory, X_train, y_train, n_folds=5)
print(f'Sklearn DecisionTreeClassifier:')
print(f'  Fold scores: {[f"{s:.4f}" for s in dtl_sk_cv["fold_scores"]]}')
print(f'  Mean Macro F1: {dtl_sk_cv["mean_f1"]:.4f} +/- {dtl_sk_cv["std_f1"]:.4f}')

In [ ]:
# Train full model & visualize tree structure
import os
dtl_full = dtl_factory()
dtl_full.fit(X_train, y_train)

tree_struct = dtl_full.get_tree_structure()
save_path = os.path.join(config.FIGURES_DIR, 'cart_tree_structure.png')
plot_tree_structure(
    tree_struct,
    feature_names=preprocessor.feature_names_,
    max_depth_display=4,
    save_path=save_path
)

In [ ]:
# Save model
save_model(dtl_full, 'cart_decision_tree.pkl')
print('Model saved.')